# Video Segment Demo

이 notebook 은 `segment` 전용 예제다.

목표:
- 폴더 안 영상 여러 개를 한 번에 읽는다.
- 각 파일의 원본 FPS 를 그대로 유지한다.
- `4분 50초 ~ 5분 50초` 구간만 잘라서 저장한다.

이번 기본 예시는 아래 경로를 기준으로 세팅되어 있다.
- `/share_ssd/ltb/Users/ltb/박스_추론용_샘플영상들/260429_서초서리풀_영건님이_프레임시간순서맞춘_3개_cctv영상들/cropped_1500sec`

권장 순서:
1. 설정 셀에서 `input_path`, `output_dir`, `start_sec`, `end_sec` 확인
2. `inspect_inputs()` 로 영상 개수/FPS/길이 확인
3. `run_segment_batch(..., dry_run=True)` 로 경로와 보고서만 먼저 생성
4. 문제가 없으면 `run_segment_batch(..., dry_run=False)` 실행


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('project root not found')

project_root = find_project_root(Path.cwd().resolve())
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from video_preprocess_workbench import load_config, inspect_inputs, run_segment_batch
from video_preprocess_workbench.pipeline import apply_overrides

print('project_root =', project_root)


## 설정 셀

가장 자주 바꾸는 값:
- `input_path`: 원본 영상 폴더 또는 파일
- `output_dir`: 잘라낸 결과 저장 폴더
- `scan_depth`: 바로 아래 파일만이면 `0`
- `start_sec`, `end_sec`: 구간 시작/종료 초

`4분 50초` 는 `290초`, `5분 50초` 는 `350초` 다.


In [ ]:
config_path = project_root / 'configs' / 'example_segment.json'
cfg = load_config(config_path)

cfg = apply_overrides(
    cfg,
    input_path='/share_ssd/ltb/Users/ltb/박스_추론용_샘플영상들/260429_서초서리풀_영건님이_프레임시간순서맞춘_3개_cctv영상들/cropped_1500sec',
    output_dir=str(project_root / 'artifacts' / 'runs' / '260429_cropped_1500sec_segment_290_350'),
    scan_depth=0,
)

# 4분 50초 ~ 5분 50초
cfg.segment.start_sec = 290.0
cfg.segment.end_sec = 350.0

# 핵심 요구사항: 파일별 원본 FPS 유지
cfg.segment.preserve_source_fps = True

# 3개 CCTV 모두 처리하므로 limit 은 보통 None 으로 둔다.
limit = None

cfg.to_dict()


## 입력 영상 점검

이 단계에서 아래를 먼저 확인한다.
- `total_files == 3` 인지
- `failed_files == 0` 인지
- FPS 와 길이가 기대와 비슷한지


In [ ]:
inspection = inspect_inputs(cfg, limit=limit)
inspection['summary']


## dry-run

실제 인코딩 없이 `segment_summary.json` 과 inventory 만 먼저 만든다.
출력 경로와 구간 설정이 맞는지 확인할 때 사용한다.


In [ ]:
dry_summary = run_segment_batch(cfg, limit=limit, dry_run=True, run_dir=inspection['run_dir'])
dry_summary


## 실제 실행

아래 셀은 필요할 때만 직접 실행하면 된다.
실행 후 결과는 `segments/` 폴더와 `reports/segment_success.csv`, `reports/segment_failed.csv` 에 남는다.


In [ ]:
# real_summary = run_segment_batch(cfg, limit=limit, dry_run=False, run_dir=inspection['run_dir'])
# real_summary
